In [3]:
import shutup; shutup.please()
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# Functions

In [ ]:
def fill_na_based_on_day(df):
    for col in df.columns:
        for i in range(1, len(df)):
            # Get the day of the week (Monday=0, Sunday=6)
            day_of_week = df.index[i].weekday()
            if pd.isna(df.iloc[i][col]):
                # For Tuesday (1) to Friday (4), and Sunday (6), replace NaN with the previous day's value
                if day_of_week in [1, 2, 3, 4, 6]:
                    if i >= 24:
                        df.iloc[i, df.columns.get_loc(col)] = df.iloc[i - 24, df.columns.get_loc(col)]
                # For Monday (0) and Saturday (5), replace NaN with the value from one week ago
                elif day_of_week in [0, 5]:
                    if i >= 24*7:
                        df.iloc[i, df.columns.get_loc(col)] = df.iloc[i - 24*7, df.columns.get_loc(col)]
    return df


def process_generation(df_name, country_name):
    df = pd.read_csv(os.path.join('data',country_name ,'Generation' , df_name), sep=',')
    df.drop('Area', axis=1, inplace=True)
    # drop columns where 'forecast' or 'consumption' is in the name in the column
    df = df.loc[:,~df.columns.str.contains('forecast', case=False)]
    df = df.loc[:,~df.columns.str.contains('consumption', case=False)]
    df = df.loc[:,~df.columns.str.contains('total load', case=False)]
    df = df.replace('n/e', np.nan)
    df = df.replace("N/A", np.nan)
    df = df.replace("", np.nan)
    df = df.replace(" ", np.nan)
    df.dropna(axis=1, how='all', inplace=True)
    df['MTU'] = df['MTU'].apply(lambda x: x[:16])
    df['MTU'] = pd.to_datetime(df['MTU'],dayfirst=True)
    df.index = df['MTU']
    df.drop('MTU', axis=1, inplace=True)
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    # remove duplicated indexes by averaging the values
    df = df.groupby(df.index).mean()
    if len(df)>370*24:
        full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq='15min')
    else:
        full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq='H')
    # Reindex the DataFrame to the full date range, filling with NaN where data is missing
    df = df.reindex(full_index)
    df = df.resample('H').mean()
    df = fill_na_based_on_day(df)
    df.columns = df.columns.str.replace('  - Actual Aggregated [MW]', ' - Actual Aggregated [MW]')

    # replace nan value with lag 24 value if index of that column is a weekday otherwise with lag 24*7 value
    return df

def generation_country(country_name):
    files = os.listdir(os.path.join('data',country_name,'Generation'))
    df_new = pd.concat(process_generation(files[i],country_name) for i in range(len(files)))
    os.makedirs(f'processed_data/{country_name}', exist_ok=True)
    df_new.to_csv(f'processed_data/{country_name}/generation.csv')
    return f"{country_name} generation data processed"

'Belgium generation data processed'

In [5]:
def price_data(df_name,country_name):
    df = pd.read_csv(os.path.join("data",country_name, 'Day_ahead_prices',df_name), sep=',')
    df['MTU (CET/CEST)'] = df['MTU (CET/CEST)'].apply(lambda x: x[:16])
    df['MTU (CET/CEST)'] = pd.to_datetime(df['MTU (CET/CEST)'],dayfirst=True)
    df.index = df['MTU (CET/CEST)']
    column_name = [col for col in df.columns if 'day-ahead' in col.lower()][0]
    df = pd.DataFrame(df[column_name], columns=[column_name])
    df.rename(columns={column_name: 'DA_price'}, inplace=True)
    df = df.replace('n/e', np.nan)
    df = df.replace("N/A", np.nan)
    df = df.replace("", np.nan)
    df = df.replace(" ", np.nan)
    df.index.name = None
    df = df.groupby(df.index).mean()
    full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq='H')
    df = df.reindex(full_index)
    df = fill_na_based_on_day(df)
    df = df.fillna(method = 'ffill')
    return df

def price_country(country_name):
    files = os.listdir(os.path.join('data',country_name,'Day_ahead_prices'))
    df_new = pd.concat(price_data(files[i],country_name) for i in range(len(files)))
    os.makedirs(f'processed_data/{country_name}', exist_ok=True)
    df_new.to_csv(f'processed_data/{country_name}/da_price.csv')
    return f"{country_name} price data processed"


In [6]:
import re
def rename_columns(df):
    # Define a function to apply to each column name
    def rename_col(col_name):
        # Use regex to find the country codes within the column name
        match = re.match(r'(.*)\((\w{2})\) > (.*)\((\w{2})\) \[MW\]', col_name)
        if match:
            # Extract the country codes and format them as "XX>>YY"
            return f"{match.group(2)}>>{match.group(4)}"
        else:
            # If the pattern doesn't match, return the column name unchanged
            return col_name
    
    # Rename the columns using the function defined above
    df.rename(columns=rename_col, inplace=True)
    return df

def process_crossborder_df(dfname, country, dirname):
    df = pd.read_csv(os.path.join('data', country, "Cross_border_flows",dirname, dfname), sep=',')
    df["Time (CET/CEST)"] = df["Time (CET/CEST)"].apply(lambda x: x[:16])
    df["Time (CET/CEST)"] = pd.to_datetime(df["Time (CET/CEST)"],dayfirst=True)
    df.index = df["Time (CET/CEST)"]
    df.drop("Time (CET/CEST)", axis=1, inplace=True)
    df = df.replace('n/e', np.nan)
    df = df.replace("N/A", np.nan)
    df = df.replace("", np.nan)
    df = df.replace(" ", np.nan)
    df.index.name = None
    df = df.groupby(df.index).mean()
    if len(df)>370*24:
        full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq='15min')
    else:
        full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq='H')
    df = df.reindex(full_index)
    df = df.resample('H').mean()
    df = rename_columns(df)
    df = fill_na_based_on_day(df)
    df = df.fillna(value=0)
    return df

def process_crossborder_dir(country,dirname):
    files = os.listdir(os.path.join('data', country, "Cross_border_flows",dirname))
    df_new = pd.concat(process_crossborder_df(files[i],country,dirname) for i in range(len(files)))
    return df_new

def process_crossborder_country(country_name):
    df_list = []
    dir_names = os.listdir(os.path.join('data', country_name, "Cross_border_flows"))
    for dirname in dir_names:
        df = process_crossborder_dir(country_name,dirname)
        df_list.append(df)
    df_new2 = pd.concat(df_list, axis=1)
    df_new2 = df_new2.fillna(value=0)
    df_new2.to_csv(os.path.join('processed_data', country_name, 'crossborder.csv'))
    return f"{country_name} crossborder data processed"

In [7]:
def demand_data(df_name, country_name):
    df = pd.read_csv(os.path.join("data",country_name, 'Demand',df_name), sep=',')
    df["Time (CET/CEST)"] = df["Time (CET/CEST)"].apply(lambda x: x[:16])
    df["Time (CET/CEST)"] = pd.to_datetime(df["Time (CET/CEST)"],dayfirst=True)
    df.index = df["Time (CET/CEST)"]
    # get that column that has actual in it
    df = df.loc[:,df.columns.str.contains('forecast', case=False)]
    # rename that one column to just demand
    df = df.rename(columns={df.columns[0]:'load_forecast'})
    df = df.replace('n/e', np.nan)
    df = df.replace("N/A", np.nan)
    df = df.replace("", np.nan)
    df = df.replace(" ", np.nan)
    df.index.name = None
    df = df.groupby(df.index).mean()
    if len(df)>370*24:
        full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq='15min')
    else:
        full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq='H')
    df = df.reindex(full_index)
    df = df.resample('H').mean()
    df = fill_na_based_on_day(df)
    df = df.fillna(method = 'ffill')
    return df

def demand_country(country_name):
    files = os.listdir(os.path.join('data',country_name,'Demand'))
    df_new = pd.concat(demand_data(files[i],country_name) for i in range(len(files)))
    os.makedirs(f'processed_data/{country_name}', exist_ok=True)
    df_new.to_csv(f'processed_data/{country_name}/load.csv')
    return f"{country_name} load forecast data processed"

In [8]:
def ws_forecast_data(df_name, country_name):
    df = pd.read_csv(os.path.join("data",country_name, 'Wind_solar_forecast',df_name), sep=',')
    df["MTU (CET/CEST)"] = df["MTU (CET/CEST)"].apply(lambda x: x[:16])
    df["MTU (CET/CEST)"] = pd.to_datetime(df["MTU (CET/CEST)"],dayfirst=True)
    df.index = df["MTU (CET/CEST)"]
    # get that column that has actual in it
    df = df.loc[:,df.columns.str.contains('day ahead', case=False)]
    # rename columns to just solar_forecast, wind_offshore_forecast, wind_onshore_forecast, by searching for the words solar, offshore, onshore to make sure to rename the right columns
    df = df.rename(columns={df.columns[df.columns.str.contains('solar', case=False)][0]:'solar_forecast'})
    df = df.rename(columns={df.columns[df.columns.str.contains('offshore', case=False)][0]:'wind_offshore_forecast'})
    df = df.rename(columns={df.columns[df.columns.str.contains('onshore', case=False)][0]:'wind_onshore_forecast'})
    df = df.replace('n/e', np.nan)
    df = df.replace("N/A", np.nan)
    df = df.replace("", np.nan)
    df = df.replace(" ", np.nan)
    df.index.name = None
    df = df.groupby(df.index).mean()
    if len(df)>370*24:
        full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq='15min')
    else:
        full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq='H')
    df = df.reindex(full_index)
    df = df.resample('H').mean()
    df = fill_na_based_on_day(df)
    return df

def ws_forecast_country(country_name):
    files = os.listdir(os.path.join('data',country_name,'Wind_solar_forecast'))
    df_new = pd.concat(ws_forecast_data(files[i],country_name) for i in range(len(files)))
    df_new = df_new.dropna(axis=1, how='all')
    df_new = df_new.fillna(0)
    os.makedirs(f'processed_data/{country_name}', exist_ok=True)
    df_new.to_csv(f'processed_data/{country_name}/ws_forecast.csv')
    return f"{country_name} wind solar forecast data processed"

# Processing

In [13]:
country_name_list = os.listdir(os.path.join('data'))

for country in tqdm(country_name_list):
    generation_country(country)
    #process_crossborder_country(country)
    #demand_country(country)
    #ws_forecast_country(country)
    #price_country(country)

100%|██████████| 22/22 [10:09<00:00, 27.69s/it]
